In [1]:
import importlib
import script
import NHL_script
import NHL_data

# Reload the script after making changes
importlib.reload(script)

# RUNNING THE SCRIPT
print("script getting date")

# DATES
date = script.get_date()

print("script getting schedule")
# SCHEDULE INFO
script.update_picture()

# NHL SCHEDULE
NHL_script.make_todays_schedule()

# NHL YESTERDAYS SCORES FROM NHL_API
NHL_script.process_yesterdays_scores_to_report()

script getting date
script getting schedule
Image found. Downloading from https://img.mlbstatic.com/mlb-images/image/upload/t_16x9/t_w1536/v1759097244/mlb/bldlnproiaocpzyzlj6g.jpg...
Image saved to data/mlb-playoffs.jpg
Today's schedule copied to NHL_data/NHL_todays_schedule.txt
Fetching fresh data for nhl_yesterdays_scores
Archived existing file to NHL_data/archived_nhl_data/nhl_yesterdays_scores_2025-10-14.json
Data saved to NHL_data/nhl_yesterdays_scores.json
Processed scores saved to NHL_data/daily_scores/NHL_scores_2025-10-14.json
report generated


In [2]:
# getting all players from the rosters to compare against skaters

import importlib
import NHL_script
importlib.reload(NHL_script)
import NHL_data
importlib.reload(NHL_data)
import file_operations
importlib.reload(file_operations)

from collections import defaultdict

teams_list = NHL_data.teams_today()
# NHL_data.update_rosters()

# ids for players on the teams playing today
roster_ids = []
for x in teams_list:
    roster = NHL_data.get_roster(x)
    for x in roster[1:]:
        roster_ids.append(x[9])

# get a list of skater paths

# Example usage
directory = "NHL_data/daily_skaters"
file_paths = NHL_data.get_sorted_skater_paths(directory)
list11 = []
for x in file_paths:
    csv_file_path = x
    file1 = file_operations.read_csv(csv_file_path)
    list11.append(file1)
    
list_of_data = []
for z in roster_ids:
    for x in list11:
        for y in x:
            if z == y[0] and y[5] == 'all':
                list_of_data.append(y)

# Combine rows by ID
combined_data = defaultdict(lambda: defaultdict(set))  # Use sets to ensure unique values

for row in list_of_data:
    player_id = row[0]
    for i, value in enumerate(row):
        if i > 6:  # Convert numeric values to floats and store in sets
            combined_data[player_id][i].add(float(value))
        else:  # Store non-numeric values in sets
            combined_data[player_id][i].add(value)

# Convert sets to sorted lists for the final output
final_data = []
for player_id, columns in combined_data.items():
    combined_row = []
    for i in range(len(list_of_data[0])):
        if i in columns:
            if i > 6:  # Sort numeric values in descending order
                combined_row.append(sorted(columns[i], reverse=True))
            else:  # Sort non-numeric values (if needed)
                combined_row.append(sorted(columns[i], reverse=True))
        else:
            combined_row.append([])
    final_data.append(combined_row)


import csv
from datetime import datetime

# Define the headers
headers = [
    "playerId", "season", "name", "team", "position", "situation", "games_played", "icetime", "shifts", "gameScore",
    "onIce_xGoalsPercentage", "offIce_xGoalsPercentage", "onIce_corsiPercentage", "offIce_corsiPercentage",
    "onIce_fenwickPercentage", "offIce_fenwickPercentage", "iceTimeRank", "I_F_xOnGoal", "I_F_xGoals", "I_F_xRebounds",
    "I_F_xFreeze", "I_F_xPlayStopped", "I_F_xPlayContinuedInZone", "I_F_xPlayContinuedOutsideZone",
    "I_F_flurryAdjustedxGoals", "I_F_scoreVenueAdjustedxGoals", "I_F_flurryScoreVenueAdjustedxGoals",
    "I_F_primaryAssists", "I_F_secondaryAssists", "I_F_shotsOnGoal", "I_F_missedShots", "I_F_blockedShotAttempts",
    "I_F_shotAttempts", "I_F_points", "I_F_goals", "I_F_rebounds", "I_F_reboundGoals", "I_F_freeze", "I_F_playStopped",
    "I_F_playContinuedInZone", "I_F_playContinuedOutsideZone", "I_F_savedShotsOnGoal", "I_F_savedUnblockedShotAttempts",
    "penalties", "I_F_penalityMinutes", "I_F_faceOffsWon", "I_F_hits", "I_F_takeaways", "I_F_giveaways",
    "I_F_lowDangerShots", "I_F_mediumDangerShots", "I_F_highDangerShots", "I_F_lowDangerxGoals",
    "I_F_mediumDangerxGoals", "I_F_highDangerxGoals", "I_F_lowDangerGoals", "I_F_mediumDangerGoals",
    "I_F_highDangerGoals", "I_F_scoreAdjustedShotsAttempts", "I_F_unblockedShotAttempts",
    "I_F_scoreAdjustedUnblockedShotAttempts", "I_F_dZoneGiveaways", "I_F_xGoalsFromxReboundsOfShots",
    "I_F_xGoalsFromActualReboundsOfShots", "I_F_reboundxGoals", "I_F_xGoals_with_earned_rebounds",
    "I_F_xGoals_with_earned_rebounds_scoreAdjusted", "I_F_xGoals_with_earned_rebounds_scoreFlurryAdjusted",
    "I_F_shifts", "I_F_oZoneShiftStarts", "I_F_dZoneShiftStarts", "I_F_neutralZoneShiftStarts", "I_F_flyShiftStarts",
    "I_F_oZoneShiftEnds", "I_F_dZoneShiftEnds", "I_F_neutralZoneShiftEnds", "I_F_flyShiftEnds", "faceoffsWon",
    "faceoffsLost", "timeOnBench", "penalityMinutes", "penalityMinutesDrawn", "penaltiesDrawn", "shotsBlockedByPlayer",
    "OnIce_F_xOnGoal", "OnIce_F_xGoals", "OnIce_F_flurryAdjustedxGoals", "OnIce_F_scoreVenueAdjustedxGoals",
    "OnIce_F_flurryScoreVenueAdjustedxGoals", "OnIce_F_shotsOnGoal", "OnIce_F_missedShots",
    "OnIce_F_blockedShotAttempts", "OnIce_F_shotAttempts", "OnIce_F_goals", "OnIce_F_rebounds",
    "OnIce_F_reboundGoals", "OnIce_F_lowDangerShots", "OnIce_F_mediumDangerShots", "OnIce_F_highDangerShots",
    "OnIce_F_lowDangerxGoals", "OnIce_F_mediumDangerxGoals", "OnIce_F_highDangerxGoals", "OnIce_F_lowDangerGoals",
    "OnIce_F_mediumDangerGoals", "OnIce_F_highDangerGoals", "OnIce_F_scoreAdjustedShotsAttempts",
    "OnIce_F_unblockedShotAttempts", "OnIce_F_scoreAdjustedUnblockedShotAttempts", "OnIce_F_xGoalsFromxReboundsOfShots",
    "OnIce_F_xGoalsFromActualReboundsOfShots", "OnIce_F_reboundxGoals", "OnIce_F_xGoals_with_earned_rebounds",
    "OnIce_F_xGoals_with_earned_rebounds_scoreAdjusted", "OnIce_F_xGoals_with_earned_rebounds_scoreFlurryAdjusted",
    "OnIce_A_xOnGoal", "OnIce_A_xGoals", "OnIce_A_flurryAdjustedxGoals", "OnIce_A_scoreVenueAdjustedxGoals",
    "OnIce_A_flurryScoreVenueAdjustedxGoals", "OnIce_A_shotsOnGoal", "OnIce_A_missedShots",
    "OnIce_A_blockedShotAttempts", "OnIce_A_shotAttempts", "OnIce_A_goals", "OnIce_A_rebounds",
    "OnIce_A_reboundGoals", "OnIce_A_lowDangerShots", "OnIce_A_mediumDangerShots", "OnIce_A_highDangerShots",
    "OnIce_A_lowDangerxGoals", "OnIce_A_mediumDangerxGoals", "OnIce_A_highDangerxGoals", "OnIce_A_lowDangerGoals",
    "OnIce_A_mediumDangerGoals", "OnIce_A_highDangerGoals", "OnIce_A_scoreAdjustedShotsAttempts",
    "OnIce_A_unblockedShotAttempts", "OnIce_A_scoreAdjustedUnblockedShotAttempts", "OnIce_A_xGoalsFromxReboundsOfShots",
    "OnIce_A_xGoalsFromActualReboundsOfShots", "OnIce_A_reboundxGoals", "OnIce_A_xGoals_with_earned_rebounds",
    "OnIce_A_xGoals_with_earned_rebounds_scoreAdjusted", "OnIce_A_xGoals_with_earned_rebounds_scoreFlurryAdjusted",
    "OffIce_F_xGoals", "OffIce_A_xGoals", "OffIce_F_shotAttempts", "OffIce_A_shotAttempts", "xGoalsForAfterShifts",
    "xGoalsAgainstAfterShifts", "corsiForAfterShifts", "corsiAgainstAfterShifts", "fenwickForAfterShifts",
    "fenwickAgainstAfterShifts"
]

# Get today's date
todays_date = datetime.now().strftime('%Y-%m-%d')

# Define the output file path
output_file = f'NHL_data/combined_skaters_{todays_date}.csv'

# Write the data to the CSV file
with open(output_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    
    # Write the headers
    writer.writerow(headers)
    
    # Write the rows
    writer.writerows(final_data)

print(f"Data has been saved to {output_file}")


Data has been saved to NHL_data/combined_skaters_2025-10-14.csv


In [3]:
import csv
import json

# Define the input CSV file and output JSON file paths
csv_file_path = 'NHL_data/combined_skaters_2025-10-14.csv'
json_file_path = 'NHL_data/combined_skaters_2025-10-14.json'

# Read the CSV file and convert it to a list of dictionaries
data = []
with open(csv_file_path, mode='r', encoding='utf-8') as csv_file:
    csv_reader = csv.DictReader(csv_file)  # Automatically uses the headers as keys
    for row in csv_reader:
        # Convert any string representations of lists back to Python lists
        for key, value in row.items():
            if value.startswith('[') and value.endswith(']'):
                try:
                    row[key] = json.loads(value)  # Parse the string as a list
                except json.JSONDecodeError:
                    pass  # Leave the value as is if it can't be parsed
        data.append(row)

# Write the data to a JSON file
with open(json_file_path, mode='w', encoding='utf-8') as json_file:
    json.dump(data, json_file, indent=4)

print(f"CSV data has been converted to JSON and saved to {json_file_path}")

CSV data has been converted to JSON and saved to NHL_data/combined_skaters_2025-10-14.json


In [4]:
import pandas as pd

def csv_to_html(csv_file_path):
    """
    Reads a CSV file and converts it to an HTML file with the same name but with a .html extension.

    Args:
        csv_file_path (str): Path to the CSV file.

    Returns:
        str: Path to the generated HTML file.
    """
    # Generate the HTML file path
    html_file_path = csv_file_path.replace('.csv', '.html')

    # Read the CSV file into a DataFrame
    df = pd.read_csv(csv_file_path)

    # Convert the DataFrame to an HTML file
    df.to_html(html_file_path, index=False)

    print(f"HTML file has been created: {html_file_path}")
    return html_file_path

# Example usage
csv_file_path = 'NHL_data/combined_skaters_2025-10-14.csv'
html_file_path = csv_to_html(csv_file_path)

HTML file has been created: NHL_data/combined_skaters_2025-10-14.html


In [6]:
import json

def read_json_to_list(json_file_path):
    """
    Reads a JSON file and returns its contents as a list of dictionaries.

    Args:
        json_file_path (str): Path to the JSON file.

    Returns:
        list: List of dictionaries containing the JSON data.
    """
    with open(json_file_path, mode='r', encoding='utf-8') as json_file:
        data = json.load(json_file)
    return data

# Example usage
json_file_path = 'NHL_data/combined_skaters_2025-10-14.json'
data = read_json_to_list(json_file_path)

# # Print the first entry to verify
# print(data[0] if data else "No data found")


# for x in data:
#     if str(x.get('playerId')) == "['8484153']":  # Convert to string for comparison
#         print(x)

# for x in data:
#     id1 = x.get('playerId')
#     print(id1)
for x in data:
    print('SOG:', x.get('I_F_shotsOnGoal'),x.get('name'), x.get('team'), 'G:', x.get('I_F_goals'),'P:', x.get('I_F_points'))

SOG: [7.0, 4.0] ['Leo Carlsson'] ['ANA'] G: [1.0, 0.0] P: [3.0, 0.0]
SOG: [0.0] ['Sam Colangelo'] ['ANA'] G: [0.0] P: [0.0]
SOG: [14.0, 5.0] ['Cutter Gauthier'] ['ANA'] G: [2.0, 0.0] P: [2.0, 0.0]
SOG: [0.0] ['Mikael Granlund'] ['ANA'] G: [0.0] P: [2.0, 0.0]
SOG: [1.0] ['Ross Johnston'] ['ANA'] G: [0.0] P: [0.0]
SOG: [11.0, 4.0] ['Alex Killorn'] ['ANA'] G: [1.0, 0.0] P: [1.0, 0.0]
SOG: [6.0, 1.0] ['Chris Kreider'] ['ANA'] G: [2.0, 0.0] P: [2.0, 0.0]
SOG: [6.0, 2.0] ['Mason McTavish'] ['ANA'] G: [0.0] P: [4.0, 1.0]
SOG: [0.0] ['Nikita Nesterenko'] ['ANA'] G: [0.0] P: [0.0]
SOG: [1.0] ['Ryan Poehling'] ['ANA'] G: [0.0] P: [0.0]
SOG: [8.0, 3.0] ['Beckett Sennecke'] ['ANA'] G: [2.0, 1.0] P: [3.0, 1.0]
SOG: [4.0, 1.0] ['Troy Terry'] ['ANA'] G: [0.0] P: [2.0, 0.0]
SOG: [4.0, 3.0] ['Frank Vatrano'] ['ANA'] G: [0.0] P: [0.0]
SOG: [4.0, 2.0] ['Radko Gudas'] ['ANA'] G: [0.0] P: [1.0, 0.0]
SOG: [2.0] ['Drew Helleson'] ['ANA'] G: [0.0] P: [0.0]
SOG: [3.0, 2.0] ['Jackson LaCombe'] ['ANA'] G: [0.0] 